# IndicBART Hindi — Persona Prompt Tuning + INT8 ONNX Quantization (v3)**What this notebook does:**1. Loads `ai4bharat/IndicBART` (mBART-style seq2seq) and freezes the backbone2. Trains 4 soft persona prefix vectors + per-persona LoRA on decoder cross-attention3. Validates quality with chrF++ score per persona4. Exports encoder + 4 persona decoders to ONNX (LoRA weights merged per persona)5. Applies INT8 dynamic quantization (~75% size reduction)6. Zips and downloads the final build for deployment**Personas:** mother · stranger · friend · gf_wife**Runtime:** GPU recommended (T4 is fine). Training takes ~25–35 min on T4.**Input files needed (upload to Colab or mount Drive):**- `hindi_mother.csv`, `hindi_stranger.csv`, `hindi_friend.csv`, `hindi_gf_wife.csv`- Each CSV must have exactly two columns: `neutral,styled`**v3 changes over v2:**- Auto-augments weak training rows (friend 60%→91%, stranger 82%→89% marker coverage)- `LORA_R` 8→16 for more adapter capacity- `EPOCHS` 40→60, `COSINE_DECAY_START_FRAC` 0.66→0.80, patience 5→8- Persona loss weights rebalanced toward worst performers (friend ×2.0, stranger ×1.6)

## 1. Setup & Constants

In [ ]:
# ── Constants ────────────────────────────────────────────────────────────────
import os

MODEL_ID   = 'ai4bharat/IndicBART'
SRC_LANG   = '<2hi>'
TGT_LANG   = '<2hi>'

PERSONAS = {0: 'mother', 1: 'stranger', 2: 'friend', 3: 'gf_wife'}
NUM_PERSONAS = len(PERSONAS)

# ── Prefix (encoder-side, per-persona soft tokens) ───────────────────────────
PREFIX_LEN   = 24
EMBED_DIM    = 1024

# ── LoRA on decoder cross-attention ──────────────────────────────────────────
# Cross-attn is where the decoder decides how much to attend to encoder content
# vs prefix. LoRA on q_proj/v_proj adds per-persona capacity to re-weight that.
LORA_R            = 16
LORA_ALPHA        = 32       # scale factor alpha/r = 2
LORA_DROPOUT      = 0.05
LORA_TARGET_TYPES = ['encoder_attn.q_proj', 'encoder_attn.v_proj']

BATCH_SIZE   = 8

# ── Training schedule ────────────────────────────────────────────────────────
EPOCHS                  = 60
WARMUP_STEPS            = 100
GRAD_CLIP               = 1.0
VAL_SPLIT               = 0.1
LR_PREFIX               = 3e-3
LR_LORA                 = 2e-4
LABEL_SMOOTHING         = 0.1
COSINE_DECAY_START_FRAC = 0.80

EARLY_STOPPING_PATIENCE = 8

MAX_SRC_LEN  = 64
MAX_TGT_LEN  = 64

# ── Per-persona loss weights ─────────────────────────────────────────────────
PERSONA_LOSS_WEIGHTS = {
    0: 0.8,     # mother   — already strong, downweight slightly
    1: 1.6,     # stranger — needs more gradient budget
    2: 2.0,     # friend   — weakest performer, biggest bump
    3: 1.0,     # gf_wife  — register is fine
}

# ── Paths ────────────────────────────────────────────────────────────────────
DATA_DIR        = '/content/data'
CKPT_DIR        = '/content/indicbart_hindi_lora'
ONNX_DIR        = '/content/indicbart_hindi_lora_onnx'
QUANT_DIR_INT8  = '/content/indicbart_hindi_lora_int8'

for d in [DATA_DIR, CKPT_DIR, ONNX_DIR, QUANT_DIR_INT8]:
    os.makedirs(d, exist_ok=True)

CSV_FILES = {
    0: f'{DATA_DIR}/hindi_mother.csv',
    1: f'{DATA_DIR}/hindi_stranger.csv',
    2: f'{DATA_DIR}/hindi_friend.csv',
    3: f'{DATA_DIR}/hindi_gf_wife.csv',
}

SPOT_CHECK = [
    'तुम कहाँ हो?',
    'खाना खा लिया?',
    'मुझे माफ करो।',
    'मैं आज देर से आऊँगा।',
    'क्या तुम ठीक हो?',
    'ऐसा फिर नहीं होगा।',
    'मेहमान कब आ रहे हैं?',
    'आपको अकेला बहुत रखता हूँ।',
    'बहुत क्रोधित हूँ।',
]

print('✅  Constants loaded.')
print(f'   LORA_R={LORA_R}  EPOCHS={EPOCHS}  patience={EARLY_STOPPING_PATIENCE}')
print(f'   Loss weights: {PERSONA_LOSS_WEIGHTS}')

In [ ]:
# ── CELL 2 — Install ALL dependencies (pinned versions) ──────────────────────
# All onnx-family packages are installed together with compatible versions
# to avoid the circular-import errors that happen when packages get layered
# in different cells across a session.

!pip install -q \
    transformers==4.44.0 \
    sentencepiece \
    sacrebleu \
    pandas \
    scikit-learn \
    matplotlib \
    peft==0.11.1

# ONNX stack — pinned to versions that work together on Python 3.12
!pip install -q \
    onnx==1.17.0 \
    onnxruntime==1.19.2 \
    onnxscript==0.1.0.dev20240817 \
    onnxconverter-common==1.14.0 \
    protobuf==3.20.3

print('✅  Cell 2 done — dependencies installed.')
print('   ⚠️  If you got an `onnx has no attribute defs` error in a previous run,')
print('       restart the runtime now (Runtime → Restart session) before continuing.')


## 2. Data Loading & Augmentation

In [ ]:
# ── CELL 3 — Upload CSV files ────────────────────────────────────────────────
# Upload your 4 CSV files using Colab's file upload widget.
# Files are saved directly into DATA_DIR.
#
# Alternative: mount Google Drive and copy files manually:
#   from google.colab import drive
#   drive.mount('/content/drive')
#   !cp /content/drive/MyDrive/hindi_*.csv {DATA_DIR}/

from google.colab import files

print('Upload your 4 CSV files: hindi_mother.csv, hindi_stranger.csv, hindi_friend.csv, hindi_gf_wife.csv')
uploaded = files.upload()

for fname, data in uploaded.items():
    dest = f'{DATA_DIR}/{fname}'
    with open(dest, 'wb') as f:
        f.write(data)
    print(f'  Saved → {dest}')


In [ ]:
# ── CELL 4 — Strip Colab's "(1)" "(2)" suffixes from re-uploaded filenames ──
import os, glob
for p in glob.glob(f'{DATA_DIR}/*(*).csv'):
    new = p.replace(' (2)', '').replace(' (1)', '').replace(' (3)', '')
    os.rename(p, new)
    print(f'  renamed → {new}')

# Verify
for p in sorted(glob.glob(f'{DATA_DIR}/*.csv')):
    print(f'  ✅ {p}')


In [ ]:
# ── CELL 5 — Verify all 4 CSVs are present and well-formed ───────────────────
import pandas as pd

all_ok = True
for pid, path in CSV_FILES.items():
    if not os.path.exists(path):
        print(f'  ❌  Missing: {path}')
        all_ok = False
    else:
        df = pd.read_csv(path)
        assert 'neutral' in df.columns and 'styled' in df.columns, \
            f'{path}: must have columns neutral,styled'
        assert len(df) >= 100, f'{path}: too few rows ({len(df)})'
        print(f'  ✅  {PERSONAS[pid]:12s}  {len(df)} rows   cols={list(df.columns)}')

if all_ok:
    print('\n✅  Cell 5 done — all CSVs verified.')
else:
    raise RuntimeError('One or more CSV files are missing. Fix before continuing.')


In [ ]:
# ── CELL 5b — Augment CSVs to strengthen register signal ─────────────────────
# Analysis of the original CSVs showed that friend and stranger personas had
# inconsistent register-marker density (friend: only 25% of rows had तू-pronoun;
# stranger: only 35% had आप). Mother and gf_wife were already at 90-100%.
#
# This cell auto-augments the weak rows ONLY, leaving well-marked rows alone.
# Augmentations are deliberately conservative:
#   - friend : append "यार!"/"भाई!" or prepend "अरे" on unmarked rows; convert
#              any leftover formal-imperatives (-कीजिए → -कर); no forced edits
#              to rows that already have a strong signal (bare-imperative +
#              informal tone counts).
#   - stranger : convert casual imperatives (करो → कीजिए); prepend "क्षमा कीजिए,"
#                or "कृपया," on unmarked rows; swap common words for sanskritic
#                equivalents where natural; leave already-formal rows alone.
#   - mother : no-op (100% coverage already).
#   - gf_wife : no-op except for the 2 rows that lacked any explicit pet-name.
#
# Expected coverage lift:
#   friend   : 60% → ~91%
#   stranger : 82% → ~89%

import os

AUGMENT_INPUT_DIR  = DATA_DIR                     # original uploaded CSVs
AUGMENT_OUTPUT_DIR = f'{DATA_DIR}/augmented'    # augmented copies
os.makedirs(AUGMENT_OUTPUT_DIR, exist_ok=True)

# ═════════════════════════════════════════════════════════════════════════════
# Augmenter (embedded inline so the notebook is self-contained)
# ═════════════════════════════════════════════════════════════════════════════
"""
Augment Hindi persona CSVs to strengthen the training signal.

For each persona, enforces ≥1 explicit register marker per row and fixes
a handful of register-inconsistency patterns. Writes to `augmented/` dir.

This is deliberately conservative — it only edits rows that need help and
leaves correct rows alone, to preserve the diversity the model needs.
"""
import pandas as pd
import re
import os
import random

random.seed(42)

# ============================================================================
# FRIEND AUGMENTER
# ============================================================================
def friend_has_strong_signal(s):
    """True if a row already has a clear friend register marker."""
    if re.search(r'तू(?![म])|तुझ|तेर', s):
        return True
    if 'यार' in s or 'भाई' in s:
        return True
    if re.search(r'\b(अरे|अबे|ओए)\b', s):
        return True
    # Bare-root imperative at clause/sentence end (खा ले!, जा!, बैठ ना!)
    if re.search(
        r'(?<![ा-ू])'   # not preceded by a vowel sign (so we match root, not inflected form)
        r'(खा|जा|सो|आ|पी|ले|दे|बोल|देख|सुन|बैठ|उठ|रख|कर|चल|रुक|मान|लौट|भाग|दौड़)'
        r'\s?(ले|ना|रे|दे)?\s*[!।?]',
        s,
    ):
        return True
    return False


def augment_friend(neutral, styled):
    result = styled

    # 1. If styled still uses तुम forms, convert (shouldn't happen in practice per
    #    our earlier check, but guard against future data)
    if re.search(r'तुम', result):
        result = result.replace('तुम्हारा', 'तेरा')
        result = result.replace('तुम्हारी', 'तेरी')
        result = result.replace('तुम्हारे', 'तेरे')
        result = result.replace('तुम्हें', 'तुझे')
        result = result.replace('तुम्हीं', 'तू ही')
        result = result.replace('तुमसे', 'तुझसे')
        result = result.replace('तुमको', 'तुझको')
        result = re.sub(r'\bतुम\b', 'तू', result)
        # Verb agreement: तुम-forms use 2p-plural, तू uses 2s-intimate
        result = re.sub(r'ते हो', 'ता है', result)
        result = re.sub(r'ती हो', 'ती है', result)
        result = re.sub(r'ते थे', 'ता था', result)
        result = re.sub(r'ती थी(?!ं)', 'ती थी', result)
        result = re.sub(r'ओगे', 'एगा', result)
        result = re.sub(r'ओगी', 'एगी', result)

    # 2. Drop any formal imperatives
    if re.search(r'(ीजिए|ाइए)', result):
        formal_to_casual = {
            'कीजिए': 'कर', 'दीजिए': 'दे', 'लीजिए': 'ले',
            'बताइए': 'बता', 'सुनिए': 'सुन', 'आइए': 'आ',
            'जाइए': 'जा', 'खाइए': 'खा', 'पीजिए': 'पी',
            'बैठिए': 'बैठ', 'उठिए': 'उठ', 'देखिए': 'देख',
            'सोइए': 'सो', 'रखिए': 'रख', 'बोलिए': 'बोल',
            'बनाइए': 'बना', 'लाइए': 'ला', 'रुकिए': 'रुक',
            'कीजिएगा': 'कर देना', 'दीजिएगा': 'दे देना',
        }
        for f, c in formal_to_casual.items():
            result = result.replace(f, c)
        result = re.sub(r'कृपया\s*,?\s*', '', result)
        result = re.sub(r'क्षमा\s+कीजिए\s*,?\s*', '', result).strip()
        result = re.sub(r'^\s*,\s*', '', result)  # clean up leading commas

    # 3. If still no explicit friend marker, append one
    if not friend_has_strong_signal(result):
        result_stripped = result.rstrip()
        end_punct = '!'
        if result_stripped and result_stripped[-1] in '।!?':
            end_punct = result_stripped[-1]
            base = result_stripped[:-1].rstrip()
        else:
            base = result_stripped

        base = base.rstrip(',').rstrip()

        # Pick marker based on what's natural. Use यार as default; use अरे as opener
        # occasionally for variety so augmentation doesn't all look identical.
        choice = random.random()
        if choice < 0.6:
            # Append "यार!" at end (most common)
            if ',' in base[-8:]:
                result = base + ' यार' + end_punct
            else:
                result = base + ', यार' + end_punct
        elif choice < 0.85:
            # Prepend "अरे"
            if base.startswith(('अरे', 'अबे', 'ओए')):
                result = base + ', यार' + end_punct
            else:
                result = 'अरे ' + base + end_punct
        else:
            # Use "भाई"
            if ',' in base[-8:]:
                result = base + ' भाई' + end_punct
            else:
                result = base + ', भाई' + end_punct

    return result


# ============================================================================
# STRANGER AUGMENTER
# ============================================================================
SANSKRITIC = ['पधार', 'भेंट', 'प्रस्थान', 'शीघ्र', 'निर्धारित', 'सूचनार्थ',
              'आवश्यकता', 'हृदय', 'व्यथित', 'रिक्त', 'उद्यान', 'भोजन',
              'शर्करा', 'अत्यंत', 'प्रसन्न', 'हर्षित', 'विश्राम', 'धन्यवाद',
              'नमस्कार', 'उचित', 'प्रश्न', 'उत्तर', 'प्रतीक्षा',
              'स्वागत', 'त्रुटि', 'क्षमायाचना', 'सम्मान', 'आदर', 'विनम्र',
              'निवेदन', 'प्रार्थना', 'कृपा', 'सहायता', 'सेवा', 'महोदया',
              'श्रीमान', 'श्रीमती', 'व्यंजन', 'विद्युत', 'द्वार', 'यंत्र',
              'मार्ग', 'रेलगाड़ी', 'अनुमान', 'ज्ञात', 'परिचय']


def stranger_has_strong_signal(s):
    if re.search(r'\bआप\b|आपको|आपका|आपकी|आपके|आपसे|आपने', s):
        return True
    if 'कृपया' in s or 'क्षमा' in s or 'महोदय' in s:
        return True
    if re.search(r'(ीजिए|ाइए|एगा)', s):
        return True
    if any(w in s for w in SANSKRITIC):
        return True
    return False


def augment_stranger(neutral, styled):
    result = styled

    # 1. Nuke intimate pronouns (rare but possible)
    if re.search(r'तू(?![म])|तुझ|तेर', result):
        result = result.replace('तुम्हारा', 'आपका').replace('तुम्हारी', 'आपकी')
        result = result.replace('तुम्हारे', 'आपके').replace('तुम्हें', 'आपको')
        result = result.replace('तुमसे', 'आपसे').replace('तुमको', 'आपको')
        result = re.sub(r'\bतुम\b', 'आप', result)
        result = re.sub(r'तू(?![म])', 'आप', result)
        result = re.sub(r'तुझे', 'आपको', result)
        result = re.sub(r'तुझ(?!े)', 'आप', result)
        result = result.replace('तेरा', 'आपका').replace('तेरी', 'आपकी')
        result = result.replace('तेरे', 'आपके')
        result = re.sub(r'ता है\b', 'ते हैं', result)
        result = re.sub(r'ती है\b', 'ती हैं', result)

    # 2. Already has a marker? done.
    if stranger_has_strong_signal(result):
        return result

    # 3. Try to convert casual imperatives to formal -इए forms
    imp_map = {
        'करो': 'कीजिए', 'दो': 'दीजिए', 'लो': 'लीजिए', 'बताओ': 'बताइए',
        'सुनो': 'सुनिए', 'आओ': 'आइए', 'जाओ': 'जाइए', 'खाओ': 'खाइए',
        'पीओ': 'पीजिए', 'पियो': 'पीजिए', 'बैठो': 'बैठिए', 'उठो': 'उठिए',
        'देखो': 'देखिए', 'सोओ': 'सोइए', 'रखो': 'रखिए', 'बोलो': 'बोलिए',
        'बनाओ': 'बनाइए', 'लाओ': 'लाइए', 'रुको': 'रुकिए',
        'समझाओ': 'समझाइए', 'बुलाओ': 'बुलाइए', 'दिखाओ': 'दिखाइए',
    }
    converted = False
    for casual, formal in imp_map.items():
        # Word-boundary match
        pattern = r'(?<![^\s])' + re.escape(casual) + r'(?=[\s।!?,]|$)'
        if re.search(pattern, result):
            result = re.sub(pattern, formal, result)
            converted = True

    # After conversion, check if it's strong enough now
    if stranger_has_strong_signal(result):
        return result

    # 4. Still no signal — prepend a formal opener. Vary the opener so we don't
    #    dump the same "क्षमा कीजिए," on every weak row.
    result_stripped = result.strip()
    is_question = result_stripped.endswith('?')
    is_imperative = bool(re.search(r'[ोए][!।]?$', result_stripped))

    choice = random.random()
    if is_question:
        # Questions → "क्षमा कीजिए," or "कृपया बताइए —"
        opener = 'क्षमा कीजिए, ' if choice < 0.6 else 'कृपया बताइए, '
        result = opener + result_stripped
    elif is_imperative or converted:
        # Imperatives → "कृपया,"
        opener = 'कृपया, ' if choice < 0.5 else 'कृपया '
        result = opener + result_stripped
    else:
        # Statement — subtler. Prepend "जी," or swap a word for its Sanskritic equivalent.
        # Safer to leave unchanged if we can't make a confident transformation.
        # Try inserting आप where meaningful (e.g., "मैं X" doesn't need आप, skip)
        if re.search(r'\b(मैं|हम)\b', result_stripped):
            return result  # 1st-person statement — leave alone
        # Otherwise prepend "जी,"
        if choice < 0.5:
            result = 'जी, ' + result_stripped
        else:
            # swap common words → sanskritic
            swaps = [
                ('पानी', 'जल'),
                ('रास्ता', 'मार्ग'),
                ('घर', 'निवास'),
                ('बच्चा', 'बालक'),
                ('बच्चे', 'बालक'),
                ('आदमी', 'व्यक्ति'),
            ]
            for a, b in swaps:
                if a in result_stripped:
                    result = result_stripped.replace(a, b, 1)
                    break
            else:
                result = 'जी, ' + result_stripped

    return result


# ============================================================================
# GF_WIFE AUGMENTER
# ============================================================================
def augment_gf_wife(neutral, styled):
    result = styled
    if re.search(r'जानू|सोना|\bजान\b|बेबी', result):
        return result

    # Append जानू at a natural spot
    result = result.rstrip()
    end_punct = '!'
    if result and result[-1] in '।!?':
        end_punct = result[-1]
        base = result[:-1].rstrip()
    else:
        base = result
    base = base.rstrip(',').rstrip()

    if ',' in base[-10:]:
        return base + ' जानू' + end_punct
    else:
        return base + ', जानू' + end_punct


# ============================================================================
# MOTHER AUGMENTER — no-op (already at 100% coverage)
# ============================================================================
def augment_mother(neutral, styled):
    return styled


# ============================================================================
# RUN
# ============================================================================
AUGMENTERS = {
    'mother':   augment_mother,
    'stranger': augment_stranger,
    'friend':   augment_friend,
    'gf_wife':  augment_gf_wife,
}


def main(input_dir='.', output_dir='augmented'):
    os.makedirs(output_dir, exist_ok=True)
    summary = []
    for name, fn in AUGMENTERS.items():
        df = pd.read_csv(f'{input_dir}/hindi_{name}.csv')
        changed = 0
        new_styled = []
        for _, row in df.iterrows():
            new_s = fn(row['neutral'], row['styled'])
            if new_s != row['styled']:
                changed += 1
            new_styled.append(new_s)
        df['styled'] = new_styled
        df.to_csv(f'{output_dir}/hindi_{name}.csv', index=False)
        summary.append((name, len(df), changed))

    print('Persona    Total   Changed  Pct')
    print('─' * 40)
    for name, total, changed in summary:
        print(f'{name:10s}  {total:5d}  {changed:5d}   {100*changed/total:.1f}%')




# ── Run the augmenter ─────────────────────────────────────────────────────────
main(input_dir=AUGMENT_INPUT_DIR, output_dir=AUGMENT_OUTPUT_DIR)

# ── Switch CSV_FILES to point at the augmented copies ─────────────────────────
CSV_FILES = {
    0: f'{AUGMENT_OUTPUT_DIR}/hindi_mother.csv',
    1: f'{AUGMENT_OUTPUT_DIR}/hindi_stranger.csv',
    2: f'{AUGMENT_OUTPUT_DIR}/hindi_friend.csv',
    3: f'{AUGMENT_OUTPUT_DIR}/hindi_gf_wife.csv',
}

# ── Verify coverage lift on primary markers ───────────────────────────────────
import pandas as pd, re

def _friend_has_signal(s):
    if re.search(r'यार|भाई|तू(?![म])|तुझ|तेर|\b(अरे|अबे|ओए)\b', s):
        return True
    return bool(re.search(
        r'(?<![ा-ू])(खा|जा|सो|आ|पी|ले|दे|बोल|देख|सुन|बैठ|उठ|रख|कर|चल|रुक|मान)'
        r'\s?(ले|ना|रे|दे)?\s*[!।?]', s
    ))

def _stranger_has_signal(s):
    if re.search(r'\bआप\b|आपको|आपका|आपकी|आपके|आपसे|आपने|कृपया|क्षमा|महोदय|(ीजिए|ाइए|एगा)', s):
        return True
    return any(w in s for w in ['पधार','भेंट','हृदय','व्यथित','उद्यान','भोजन','विद्युत','द्वार','व्यंजन','मार्ग','नमस्कार','अत्यंत','हर्षित'])

print('\n── Coverage check after augmentation ──')
for pid, path in CSV_FILES.items():
    df = pd.read_csv(path)
    name = PERSONAS[pid]
    if name == 'friend':
        pct = 100 * df['styled'].apply(_friend_has_signal).mean()
        print(f'  {name:10s}  any-friend-signal  : {pct:.0f}%  (target ≥80%)')
    elif name == 'stranger':
        pct = 100 * df['styled'].apply(_stranger_has_signal).mean()
        print(f'  {name:10s}  any-formal-signal  : {pct:.0f}%  (target ≥85%)')
    elif name == 'mother':
        pct = 100 * df['styled'].str.contains('माँ').mean()
        print(f'  {name:10s}  has माँ            : {pct:.0f}%  (target 100%)')
    elif name == 'gf_wife':
        pct = 100 * df['styled'].str.contains(r'जानू|सोना|\bजान\b|बेबी', regex=True).mean()
        print(f'  {name:10s}  has gf-marker      : {pct:.0f}%  (target ≥95%)')

print('\n✅  Cell 5b done — augmented CSVs written; CSV_FILES now points at them.')
print(f'   Original files still in : {AUGMENT_INPUT_DIR}')
print(f'   Augmented files in      : {AUGMENT_OUTPUT_DIR}')


## 3. Model & Training

In [ ]:
# ── CELL 6 — Load tokenizer + backbone model ─────────────────────────────────
# The backbone is loaded in FP32 and immediately frozen.
# Only the prefix embeddings + LoRA adapters will be trained.

import torch
from transformers import AutoTokenizer, MBartForConditionalGeneration

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')

# ── Tokenizer ────────────────────────────────────────────────────────────────
print('Loading tokenizer...')
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID,
    src_lang=SRC_LANG,
    tgt_lang=TGT_LANG,
    trust_remote_code=True,
)
print(f'  Vocab size : {tokenizer.vocab_size:,}')

# ── Backbone ─────────────────────────────────────────────────────────────────
print('Loading IndicBART backbone...')
backbone = MBartForConditionalGeneration.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float32,
    trust_remote_code=True,
).to(DEVICE)

for param in backbone.parameters():
    param.requires_grad = False

backbone.eval()

total_params  = sum(p.numel() for p in backbone.parameters())
frozen_params = sum(p.numel() for p in backbone.parameters() if not p.requires_grad)
print(f'  Total params  : {total_params:,}')
print(f'  Frozen params : {frozen_params:,}  (100% — backbone fully frozen)')
print()
print('✅  Cell 6 done — backbone loaded and frozen.')


In [ ]:
# ── CELL 7 — Define PersonaLoRAPrefixModel ───────────────────────────────────
#
# Architecture = prefix (encoder-side, per-persona soft tokens)
#              + LoRA (decoder cross-attention, per-persona low-rank adapters)
#
# Why this split:
#   - Prefix controls STYLE/REGISTER — what vocabulary to use (माँ, जानू, आप)
#   - LoRA on cross-attention controls CONTENT ATTENTION — how well the decoder
#     attends to input tokens vs the prefix. Fixes semantic drift.

import torch
import torch.nn as nn
import torch.nn.functional as F


class PerPersonaLoRA(nn.Module):
    """
    Wraps a frozen nn.Linear with per-persona LoRA adapters.
    For each persona p: output = base(x) + scale * (x @ A[p] @ B[p])

    Active persona is set via a buffer (so it propagates through hooks correctly),
    but we read it as a Python int only once per `set_persona` call, not per
    forward — keeps it cheap.
    """
    def __init__(self, base_linear, num_personas, r, alpha, dropout=0.0):
        super().__init__()
        self.base = base_linear
        in_f  = base_linear.in_features
        out_f = base_linear.out_features
        self.r = r
        self.scale = alpha / r
        self.num_personas = num_personas

        self.A = nn.Parameter(torch.zeros(num_personas, in_f, r))
        self.B = nn.Parameter(torch.zeros(num_personas, r, out_f))
        nn.init.kaiming_uniform_(self.A, a=5 ** 0.5)
        # B stays zero so LoRA starts as identity

        self.drop = nn.Dropout(dropout) if dropout > 0 else nn.Identity()

        # Cached active persona — a plain Python int, set by set_persona()
        self._active_pid = 0

    def set_persona(self, pid):
        """Set active persona. pid: int."""
        self._active_pid = int(pid) if not isinstance(pid, int) else pid

    def forward(self, x):
        base_out = self.base(x)
        pid = self._active_pid
        A = self.A[pid]   # (in_f, r)
        B = self.B[pid]   # (r, out_f)
        lora_out = self.drop(x) @ A @ B
        return base_out + self.scale * lora_out


def _inject_lora(backbone, num_personas, r, alpha, dropout, target_suffixes):
    """
    Replace target Linear layers in backbone with PerPersonaLoRA wrappers.
    Returns list of (full_name, wrapped_module) for the injected modules.
    """
    injected = []

    def _should_target(name):
        return any(name.endswith(suf) for suf in target_suffixes)

    for name, module in list(backbone.named_modules()):
        for child_name, child in list(module.named_children()):
            full_name = f'{name}.{child_name}' if name else child_name
            if isinstance(child, nn.Linear) and _should_target(full_name):
                wrapped = PerPersonaLoRA(
                    base_linear=child,
                    num_personas=num_personas,
                    r=r, alpha=alpha, dropout=dropout,
                )
                for p in wrapped.base.parameters():
                    p.requires_grad = False
                setattr(module, child_name, wrapped)
                injected.append((full_name, wrapped))

    return injected


class PersonaLoRAPrefixModel(nn.Module):
    def __init__(self, backbone, num_personas, prefix_len, embed_dim,
                 lora_r, lora_alpha, lora_dropout, lora_target_suffixes,
                 label_smoothing=0.0, persona_loss_weights=None, pad_id=0):
        super().__init__()
        self.backbone     = backbone
        self.prefix_len   = prefix_len
        self.embed_dim    = embed_dim
        self.num_personas = num_personas
        self.label_smoothing = label_smoothing
        self.pad_id       = pad_id

        self.persona_embeddings = nn.Embedding(
            num_personas, prefix_len * embed_dim,
        )
        nn.init.normal_(self.persona_embeddings.weight, mean=0.0, std=0.05)

        self.lora_modules = _inject_lora(
            backbone, num_personas, lora_r, lora_alpha,
            lora_dropout, lora_target_suffixes,
        )
        print(f'Injected LoRA into {len(self.lora_modules)} modules:')
        for name, _ in self.lora_modules:
            print(f'  - {name}')

        if persona_loss_weights is None:
            w = torch.ones(num_personas)
        else:
            w = torch.tensor([persona_loss_weights[i] for i in range(num_personas)],
                              dtype=torch.float32)
        self.register_buffer('persona_weights', w)

    def _set_active_persona(self, pid):
        """Set the active persona on ALL LoRA modules. pid: int."""
        for _, mod in self.lora_modules:
            mod.set_persona(pid)

    def get_prefix(self, persona_ids):
        flat = self.persona_embeddings(persona_ids)
        return flat.view(-1, self.prefix_len, self.embed_dim)

    def forward(self, input_ids, attention_mask, labels, persona_ids):
        device = input_ids.device

        total_loss   = torch.zeros(1, device=device)
        total_weight = torch.zeros(1, device=device)

        unique_pids = persona_ids.unique().tolist()

        for pid in unique_pids:
            mask = (persona_ids == pid)
            idx  = mask.nonzero(as_tuple=True)[0]
            if len(idx) == 0: continue

            self._set_active_persona(pid)

            sub_ids     = input_ids[idx]
            sub_mask    = attention_mask[idx]
            sub_labels  = labels[idx]
            sub_pids    = persona_ids[idx]

            token_embeds = self.backbone.model.shared(sub_ids)
            prefix       = self.get_prefix(sub_pids)
            encoder_inputs = torch.cat([prefix, token_embeds], dim=1)

            prefix_mask = torch.ones(
                sub_mask.size(0), self.prefix_len,
                dtype=sub_mask.dtype, device=device,
            )
            extended_mask = torch.cat([prefix_mask, sub_mask], dim=1)

            encoder_outputs = self.backbone.model.encoder(
                inputs_embeds=encoder_inputs,
                attention_mask=extended_mask,
            )

            decoder_input_ids = self.backbone.prepare_decoder_input_ids_from_labels(sub_labels)
            outputs = self.backbone(
                attention_mask    = extended_mask,
                encoder_outputs   = encoder_outputs,
                decoder_input_ids = decoder_input_ids,
                labels            = None,
            )
            logits = outputs.logits

            flat_logits = logits.reshape(-1, logits.size(-1))
            flat_labels = sub_labels.reshape(-1)
            per_token_loss = F.cross_entropy(
                flat_logits, flat_labels,
                ignore_index=-100,
                label_smoothing=self.label_smoothing,
                reduction='none',
            ).view(sub_labels.size(0), sub_labels.size(1))
            valid = (sub_labels != -100).float()
            per_ex = (per_token_loss * valid).sum(dim=1) / valid.sum(dim=1).clamp_min(1.0)

            w = self.persona_weights[pid]
            sub_weight = w * len(idx)
            total_loss   = total_loss   + per_ex.sum() * w
            total_weight = total_weight + sub_weight

        return total_loss / total_weight.clamp_min(1.0)


# ── Instantiate ───────────────────────────────────────────────────────────────
model = PersonaLoRAPrefixModel(
    backbone=backbone,
    num_personas=NUM_PERSONAS,
    prefix_len=PREFIX_LEN,
    embed_dim=EMBED_DIM,
    lora_r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    lora_target_suffixes=LORA_TARGET_TYPES,
    label_smoothing=LABEL_SMOOTHING,
    persona_loss_weights=PERSONA_LOSS_WEIGHTS,
    pad_id=tokenizer.pad_token_id,
).to(DEVICE)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
prefix_params = model.persona_embeddings.weight.numel()
lora_params   = trainable - prefix_params
print()
print(f'Trainable params : {trainable:,}')
print(f'  prefix params : {prefix_params:,}')
print(f'  LoRA params   : {lora_params:,}  ({len(model.lora_modules)} modules × per-persona)')
print()
print('✅  Cell 7 done — PersonaLoRAPrefixModel defined.')


In [ ]:
# ── CELL 8 — Dataset + DataLoader ────────────────────────────────────────────
# Loads all 4 CSVs, assigns persona_id per row, splits 90/10 train/val.
# Tokenization happens at collate time (dynamic padding per batch).

import pandas as pd
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split


class PersonaDataset(Dataset):
    def __init__(self, rows):
        self.rows = rows

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, idx):
        neutral, styled, pid = self.rows[idx]
        return neutral, styled, pid


def collate_fn(batch):
    neutrals, styleds, pids = zip(*batch)

    enc = tokenizer(
        list(neutrals),
        max_length=MAX_SRC_LEN,
        padding=True,
        truncation=True,
        return_tensors='pt',
    )

    dec = tokenizer(
        text_target=list(styleds),
        max_length=MAX_TGT_LEN,
        padding=True,
        truncation=True,
        return_tensors='pt',
    )

    labels = dec['input_ids'].clone()
    labels[labels == tokenizer.pad_token_id] = -100

    return {
        'input_ids'      : enc['input_ids'],
        'attention_mask' : enc['attention_mask'],
        'labels'         : labels,
        'persona_ids'    : torch.tensor(pids, dtype=torch.long),
    }


# ── Load CSVs ─────────────────────────────────────────────────────────────────
all_rows   = []
train_rows = []
val_rows   = []

for pid, path in CSV_FILES.items():
    df = pd.read_csv(path).dropna()
    df = df[df['neutral'].str.strip() != '']
    df = df[df['styled'].str.strip()  != '']
    rows = [(r['neutral'], r['styled'], pid) for _, r in df.iterrows()]

    tr, vl = train_test_split(rows, test_size=VAL_SPLIT, random_state=42)
    train_rows.extend(tr)
    val_rows.extend(vl)
    print(f'  {PERSONAS[pid]:12s}  total={len(rows)}  train={len(tr)}  val={len(vl)}')

train_ds = PersonaDataset(train_rows)
val_ds   = PersonaDataset(val_rows)

train_dl = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  collate_fn=collate_fn, num_workers=2)
val_dl   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn, num_workers=2)

print(f'\nTrain batches : {len(train_dl)}')
print(f'Val   batches : {len(val_dl)}')
print()
print('✅  Cell 8 done — datasets ready.')


In [ ]:
# ── CELL 9 — Training loop ───────────────────────────────────────────────────
# Two parameter groups with different LRs:
#   - Prefix embeddings : LR_PREFIX (3e-3)
#   - LoRA (A, B)       : LR_LORA   (2e-4) — ~15× smaller (more params)

import math
import torch
from torch.optim import AdamW
from torch.optim.lr_scheduler import LambdaLR

prefix_params = [model.persona_embeddings.weight]
lora_params   = []
for _, mod in model.lora_modules:
    lora_params.append(mod.A)
    lora_params.append(mod.B)

optimizer = AdamW([
    {'params': prefix_params, 'lr': LR_PREFIX, 'weight_decay': 0.01},
    {'params': lora_params,   'lr': LR_LORA,   'weight_decay': 0.0 },
])

total_steps  = len(train_dl) * EPOCHS
decay_start  = int(total_steps * COSINE_DECAY_START_FRAC)

def lr_lambda(step):
    if step < WARMUP_STEPS:
        return float(step) / max(1, WARMUP_STEPS)
    if step < decay_start:
        return 1.0
    progress = (step - decay_start) / max(1, total_steps - decay_start)
    return 0.1 + 0.9 * 0.5 * (1.0 + math.cos(math.pi * progress))

scheduler = LambdaLR(optimizer, lr_lambda)

patience = EARLY_STOPPING_PATIENCE
no_improve = 0
best_epoch = 0
best_val_loss = float('inf')
global_step   = 0

train_losses = []
val_losses   = []

print(f'Training for {EPOCHS} epochs  |  {total_steps} total steps')
print(f'Prefix LR: {LR_PREFIX}  |  LoRA LR: {LR_LORA}')
print(f'Trainable params: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}')
print()

for epoch in range(1, EPOCHS + 1):
    model.train()
    train_loss_sum = 0.0

    for step, batch in enumerate(train_dl, 1):
        batch = {k: v.to(DEVICE) for k, v in batch.items()}
        loss = model(
            input_ids      = batch['input_ids'],
            attention_mask = batch['attention_mask'],
            labels         = batch['labels'],
            persona_ids    = batch['persona_ids'],
        )
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(
            [p for p in model.parameters() if p.requires_grad],
            GRAD_CLIP,
        )
        optimizer.step()
        scheduler.step()

        train_loss_sum += loss.item()
        global_step += 1

        if step % 50 == 0:
            avg = train_loss_sum / step
            lr_p = optimizer.param_groups[0]['lr']
            lr_l = optimizer.param_groups[1]['lr']
            print(f'  Epoch {epoch}  step {step:4d}/{len(train_dl)}  '
                  f'loss={avg:.4f}  lr_prefix={lr_p:.2e}  lr_lora={lr_l:.2e}')

    train_avg = train_loss_sum / len(train_dl)
    train_losses.append(train_avg)

    model.eval()
    val_loss_sum = 0.0
    with torch.no_grad():
        for batch in val_dl:
            batch = {k: v.to(DEVICE) for k, v in batch.items()}
            loss = model(
                input_ids      = batch['input_ids'],
                attention_mask = batch['attention_mask'],
                labels         = batch['labels'],
                persona_ids    = batch['persona_ids'],
            )
            val_loss_sum += loss.item()
    val_avg = val_loss_sum / len(val_dl)
    val_losses.append(val_avg)

    marker = '  ← best' if val_avg < best_val_loss else ''
    print(f'Epoch {epoch}/{EPOCHS}  train_loss={train_avg:.4f}  val_loss={val_avg:.4f}{marker}')

    if val_avg < best_val_loss:
        best_val_loss = val_avg
        best_epoch = epoch
        no_improve = 0

        import numpy as np
        prefix_weights = model.persona_embeddings.weight.detach().cpu().numpy()
        np.save(f'{CKPT_DIR}/persona_prefixes_best.npy', prefix_weights)

        lora_state = {}
        for name, mod in model.lora_modules:
            lora_state[f'{name}.A'] = mod.A.detach().cpu()
            lora_state[f'{name}.B'] = mod.B.detach().cpu()
        torch.save(lora_state, f'{CKPT_DIR}/lora_weights_best.pt')
        print(f'  Saved best checkpoint  (val_loss={best_val_loss:.4f})')
    else:
        no_improve += 1
        print(f'  No improvement for {no_improve} epoch(s)')
        if no_improve >= patience:
            print(f'\n🛑 Early stopping triggered at epoch {epoch}')
            break
    print()

print(f'✅ Training complete')
print(f'🏆 Best epoch: {best_epoch}  |  Best val_loss={best_val_loss:.4f}')


### Save / Restore Checkpoints (Google Drive)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import shutil, os

# Create a save folder on your Drive
SAVE_DIR = '/content/drive/MyDrive/indicbart_hindi_lora_trained'
os.makedirs(SAVE_DIR, exist_ok=True)

# Copy the best checkpoint files (saved during training by Cell 9)
shutil.copy(f'{CKPT_DIR}/persona_prefixes_best.npy', f'{SAVE_DIR}/persona_prefixes_best.npy')
shutil.copy(f'{CKPT_DIR}/lora_weights_best.pt',     f'{SAVE_DIR}/lora_weights_best.pt')

# Sanity check
for f in os.listdir(SAVE_DIR):
    sz = os.path.getsize(f'{SAVE_DIR}/{f}') / 1e6
    print(f'  {f}  {sz:.1f} MB')

print('\n✅ Saved to Drive. Safe to close the laptop now.')

In [ ]:
# ── CELL 10 — Loss curves ────────────────────────────────────────────────────
import matplotlib.pyplot as plt

plt.figure()
plt.plot(train_losses, label='Train Loss')
plt.plot(val_losses,   label='Validation Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Training vs Validation Loss')
plt.legend()
plt.grid()
plt.show()


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import shutil

SAVE_DIR = '/content/drive/MyDrive/indicbart_hindi_lora_trained'

# Copy checkpoints from Drive back into CKPT_DIR so Cell 11 finds them
shutil.copy(f'{SAVE_DIR}/persona_prefixes_best.npy', f'{CKPT_DIR}/persona_prefixes_best.npy')
shutil.copy(f'{SAVE_DIR}/lora_weights_best.pt',     f'{CKPT_DIR}/lora_weights_best.pt')
print('✅ Restored trained weights from Drive')

## 4. Evaluation

In [ ]:
# ── CELL 11 — PyTorch inference (loads best weights, defines rewrite()) ──────
# Must set active persona on all LoRA modules BEFORE calling generate(),
# since LoRA routing uses each module's _active_pid.

import numpy as np
import torch
import re

# ── Reload best weights ───────────────────────────────────────────────────────
best_prefix = np.load(f'{CKPT_DIR}/persona_prefixes_best.npy')
model.persona_embeddings.weight.data = torch.tensor(best_prefix).to(DEVICE)

lora_state = torch.load(f'{CKPT_DIR}/lora_weights_best.pt')
for name, mod in model.lora_modules:
    mod.A.data = lora_state[f'{name}.A'].to(DEVICE)
    mod.B.data = lora_state[f'{name}.B'].to(DEVICE)
model.eval()
print('Best prefix + LoRA weights loaded.')

# ── Find the Hindi target language token ──────────────────────────────────────
TGT_LANG_ID = tokenizer.convert_tokens_to_ids(TGT_LANG)
print(f'Target lang token: {TGT_LANG!r}  →  id: {TGT_LANG_ID}')

# ── Junk-token blocklist (non-Devanagari noise from training data) ────────────
def _has_devanagari(s):
    return any('\u0900' <= ch <= '\u097f' for ch in s)
def _is_junk_token(tok_str):
    if _has_devanagari(tok_str): return False
    if tok_str.lstrip('▁').startswith(('http', '#', '@')): return True
    return False
vocab = tokenizer.get_vocab()
bad_words_ids = [[tok_id] for tok_str, tok_id in vocab.items() if _is_junk_token(tok_str)]
print(f'Blocked {len(bad_words_ids):,} junk tokens')

# Per-persona generation config
GEN_CONFIG = {
    0: dict(num_beams=4, length_penalty=0.8, repetition_penalty=1.2, no_repeat_ngram_size=3),
    1: dict(num_beams=5, length_penalty=1.0, repetition_penalty=1.1, no_repeat_ngram_size=3),
    2: dict(num_beams=4, length_penalty=0.7, repetition_penalty=1.4, no_repeat_ngram_size=3),
    3: dict(num_beams=4, length_penalty=0.7, repetition_penalty=1.5, no_repeat_ngram_size=2),
}

def _devanagari_only(text):
    return ''.join(
        ch for ch in text
        if '\u0900' <= ch <= '\u097f' or ch in ' \t।?!,'
    ).strip()

def _first_sentence(text):
    m = re.search(r'[।?!]', text)
    return text[:m.end()].strip() if m else text.strip()

def _adaptive_max_new_tokens(text, persona_id):
    input_tokens = len(tokenizer.encode(text, add_special_tokens=False))
    multiplier = 2.2 if persona_id in (0, 3) else 2.0
    return max(15, min(35, int(input_tokens * multiplier) + 5))


def rewrite(text, persona_id, max_new_tokens=None):
    if max_new_tokens is None:
        max_new_tokens = _adaptive_max_new_tokens(text, persona_id)

    enc = tokenizer(
        [f'{SRC_LANG} {text}'],
        max_length=MAX_SRC_LEN, truncation=True, return_tensors='pt',
    ).to(DEVICE)

    pid_tensor = torch.tensor([persona_id], dtype=torch.long, device=DEVICE)

    # 🔥 LoRA-specific: set active persona on all adapter modules
    model._set_active_persona(persona_id)

    with torch.no_grad():
        token_embeds = backbone.model.shared(enc['input_ids'])
        prefix       = model.get_prefix(pid_tensor)
        enc_inputs   = torch.cat([prefix, token_embeds], dim=1)

        prefix_mask  = torch.ones(1, PREFIX_LEN, dtype=torch.long, device=DEVICE)
        ext_mask     = torch.cat([prefix_mask, enc['attention_mask']], dim=1)

        encoder_out  = backbone.model.encoder(
            inputs_embeds=enc_inputs,
            attention_mask=ext_mask,
        )

        output_ids = backbone.generate(
            attention_mask      = ext_mask,
            encoder_outputs     = encoder_out,
            forced_bos_token_id = TGT_LANG_ID,
            max_new_tokens      = max_new_tokens,
            min_new_tokens      = 3,
            early_stopping      = True,
            bad_words_ids       = bad_words_ids,
            renormalize_logits  = True,
            **GEN_CONFIG[persona_id],
        )

    raw   = tokenizer.decode(output_ids[0], skip_special_tokens=True)
    clean = _devanagari_only(raw)
    return _first_sentence(clean)


# ── Spot-check ────────────────────────────────────────────────────────────────
print('\n── Spot-check: neutral Hindi → all personas ──\n')
for neutral in SPOT_CHECK:
    print(f'INPUT : {neutral}')
    for pid, pname in PERSONAS.items():
        out = rewrite(neutral, pid)
        print(f'  [{pname:12s}]  {out}')
    print()

print('✅  Cell 11 done — generation complete')


In [ ]:
# ── CELL 12 — chrF++ validation per persona ──────────────────────────────────
# chrF++ is preferred over BLEU for morphologically rich languages like Hindi.
# Score above 45 is acceptable for style transfer.

import sacrebleu
from collections import defaultdict

model.eval()

per_persona_hyps = defaultdict(list)
per_persona_refs = defaultdict(list)

print('Running validation inference...')

for neutral, styled, pid in val_rows:
    hyp = rewrite(neutral, pid)
    per_persona_hyps[pid].append(hyp)
    per_persona_refs[pid].append(styled)

print()
print('── chrF++ per persona (PyTorch baseline) ──')
all_hyps = []
all_refs = []
all_pass = True
PT_BASELINE_CHRF = {}   # save for the comparison cell later

for pid, pname in PERSONAS.items():
    hyps = per_persona_hyps[pid]
    refs = per_persona_refs[pid]
    chrf = sacrebleu.corpus_chrf(hyps, [refs]).score
    PT_BASELINE_CHRF[pid] = chrf
    status = '✅' if chrf >= 45.0 else '⚠️  (below 45 — consider more epochs or data)'
    print(f'  {pname:14s}  chrF++={chrf:.2f}  {status}')
    if chrf < 45.0:
        all_pass = False
    all_hyps.extend(hyps)
    all_refs.extend(refs)

PT_BASELINE_OVERALL = sacrebleu.corpus_chrf(all_hyps, [all_refs]).score
print(f'\n  Overall        chrF++={PT_BASELINE_OVERALL:.2f}')

if all_pass:
    print('\n✅  Cell 12 done — quality gate passed.')
else:
    print('\n⚠️  Some personas below threshold. Continue to export but consider retraining.')


In [ ]:
# ── CELL 13 — Diagnostic: inspect predictions for ALL personas on val set ────
import random
from collections import Counter

random.seed(0)

N_SAMPLES = 20
SEMANTIC_JACCARD_FLOOR = 0.08

PERSONA_MARKERS = {
    0: {
        'name'   : 'mother',
        'expect' : ['माँ', 'आप'],
        'forbid' : ['जानू', 'सोना', 'यार', 'भाई', ' तू ', ' तू?', ' तू।', ' तू!'],
    },
    1: {
        'name'   : 'stranger',
        'expect' : ['आप', 'कृपया', 'क्षमा', 'महोदय', 'जी', 'धन्यवाद', 'शुभ'],
        'forbid' : ['माँ', 'जानू', 'सोना', 'यार', 'भाई',
                    ' तू ', ' तू?', ' तू।', ' तू!'],
    },
    2: {
        'name'   : 'friend',
        'expect' : ['यार', 'भाई', ' तू ', ' तू?', ' तू।', ' तू!', 'तुझे', 'तेरा', 'तेरी',
                    ' ले!', ' ले।', ' ले ', 'खा ले', 'कर ले', 'जा ना', 'बता ना'],
        'forbid' : ['माँ', 'जानू', 'सोना', 'कृपया', 'क्षमा', 'महोदय', 'आप'],
    },
    3: {
        'name'   : 'gf_wife',
        'expect' : ['जानू', 'सोना', 'जान', 'तुम', 'बेबी'],
        'forbid' : ['माँ', 'यार', 'भाई', 'कृपया', 'क्षमा', 'महोदय',
                    ' तू ', ' तू?', ' तू।', ' तू!'],
    },
}

def contains_any(text, markers):
    padded = f' {text} '
    return any(m in padded if m.startswith(' ') or m.endswith(' ') else m in text
               for m in markers)

def classify(pred, ref, persona_id):
    flags = []
    markers = PERSONA_MARKERS[persona_id]
    if contains_any(pred, markers['forbid']):
        flags.append('❌REGISTER_LEAK')
    if contains_any(pred, markers['expect']):
        flags.append('✓REGISTER_OK')
    if len(pred) < 0.5 * len(ref):
        flags.append('⚠️TRUNCATED')
    if len(pred) > 1.8 * len(ref):
        flags.append('⚠️TOO_LONG')
    return flags

def char_jaccard(a, b):
    def trigrams(s):
        s = ''.join(s.split())
        return {s[i:i+3] for i in range(len(s)-2)}
    A, B = trigrams(a), trigrams(b)
    if not A or not B: return 0.0
    return len(A & B) / len(A | B)

overall_summary = {}

for pid in sorted(PERSONAS.keys()):
    pname = PERSONAS[pid]
    print('\n' + '█' * 100)
    print(f'  PERSONA: {pname.upper()}  (id={pid})')
    print('█' * 100)

    persona_val = [(n, s, p) for (n, s, p) in val_rows if p == pid]
    sample = random.sample(persona_val, min(N_SAMPLES, len(persona_val)))

    all_flags = Counter()
    jaccards  = []
    semantic_fails = 0

    for i, (neutral, reference, _) in enumerate(sample, 1):
        pred = rewrite(neutral, pid)
        flags = classify(pred, reference, pid)
        jac = char_jaccard(pred, reference)
        jaccards.append(jac)

        is_semantic_fail = (
            jac < SEMANTIC_JACCARD_FLOOR
            and '✓REGISTER_OK' in flags
            and '❌REGISTER_LEAK' not in flags
        )
        if is_semantic_fail:
            flags.append('❌SEMANTIC_DRIFT')
            semantic_fails += 1

        all_flags.update(flags)

        print(f'{i:>3}. IN : {neutral}')
        print(f'     REF: {reference}')
        print(f'     OUT: {pred}')
        print(f'     {" ".join(flags) if flags else "(clean)"}   jac={jac:.2f}')
        print()

    n = len(sample)
    register_leak = all_flags.get('❌REGISTER_LEAK', 0)
    register_ok   = all_flags.get('✓REGISTER_OK', 0)
    truncated     = all_flags.get('⚠️TRUNCATED', 0)
    too_long      = all_flags.get('⚠️TOO_LONG', 0)
    mean_jac      = sum(jaccards) / len(jaccards)

    print('─' * 100)
    print(f'  {pname} summary:')
    print(f'    register correct      : {register_ok}/{n}  ({100*register_ok/n:.0f}%)')
    print(f'    register leaks        : {register_leak}/{n}  ({100*register_leak/n:.0f}%)')
    print(f'    likely semantic drift : {semantic_fails}/{n}  ({100*semantic_fails/n:.0f}%)')
    print(f'    truncated/too-long    : {truncated}/{n} / {too_long}/{n}')
    print(f'    mean char-trigram jac : {mean_jac:.3f}')

    if register_leak >= 3:
        verdict = '❌ REGISTER PROBLEM'
    elif semantic_fails >= 4:
        verdict = '❌ SEMANTIC PROBLEM'
    elif register_leak + semantic_fails >= 4:
        verdict = '⚠️ MIXED ERRORS'
    elif register_ok >= 0.75 * n and semantic_fails <= 2:
        verdict = '✅ LOOKS GOOD'
    else:
        verdict = '⚠️ BORDERLINE — read samples by eye'

    print(f'    verdict: {verdict}')

    overall_summary[pid] = {
        'name'            : pname,
        'register_ok'     : register_ok,
        'register_leak'   : register_leak,
        'semantic_drift'  : semantic_fails,
        'truncated'       : truncated,
        'too_long'        : too_long,
        'mean_jaccard'    : mean_jac,
        'verdict'         : verdict,
        'n'               : n,
    }

print('\n\n' + '═' * 100)
print('  OVERALL COMPARISON')
print('═' * 100)
print(f'{"persona":<12} {"reg_ok":<10} {"reg_leak":<10} {"sem_drift":<12} {"mean_jac":<10} verdict')
print('─' * 100)
for pid, s in overall_summary.items():
    print(f'{s["name"]:<12} '
          f'{s["register_ok"]}/{s["n"]:<8} '
          f'{s["register_leak"]}/{s["n"]:<8} '
          f'{s["semantic_drift"]}/{s["n"]:<10} '
          f'{s["mean_jaccard"]:<10.3f} '
          f'{s["verdict"]}')


## 5. ONNX Export & INT8 Quantization

In [ ]:
!pip install -q onnx onnxruntime

In [ ]:
# ── CELL 14 — Export shared encoder to ONNX ──────────────────────────────────
# One encoder shared across all 4 personas; persona_prefixes.npy is loaded
# at runtime and prepended to the token embeddings inside the graph.

import os, glob
import torch
import torch.nn as nn
import onnx
import numpy as np

model.eval()
backbone.eval()

# Wipe old files
for path in glob.glob(f'{ONNX_DIR}/*.onnx') + glob.glob(f'{ONNX_DIR}/*.onnx.data'):
    os.remove(path)
    print(f'  removed {os.path.basename(path)}')

# Save prefixes (3D: persona × prefix_len × embed_dim)
prefix_weights = model.persona_embeddings.weight.detach().cpu().numpy()
prefix_weights_3d = prefix_weights.reshape(NUM_PERSONAS, PREFIX_LEN, EMBED_DIM)
np.save(f'{ONNX_DIR}/persona_prefixes.npy', prefix_weights_3d)
print(f'\nSaved persona_prefixes.npy  shape={prefix_weights_3d.shape}')


class SharedEncoder(nn.Module):
    def __init__(self, backbone, prefix_len):
        super().__init__()
        self.backbone   = backbone
        self.prefix_len = prefix_len
    def forward(self, input_ids, attention_mask, prefix_embeds):
        token_embeds = self.backbone.model.shared(input_ids)
        enc_inputs   = torch.cat([prefix_embeds, token_embeds], dim=1)
        B = input_ids.size(0)
        prefix_mask = torch.ones(B, self.prefix_len,
                                 dtype=attention_mask.dtype,
                                 device=attention_mask.device)
        ext_mask = torch.cat([prefix_mask, attention_mask], dim=1)
        out = self.backbone.model.encoder(
            inputs_embeds=enc_inputs,
            attention_mask=ext_mask,
        )
        return out.last_hidden_state, ext_mask


shared_encoder = SharedEncoder(backbone, PREFIX_LEN).to(DEVICE).eval()

dummy_ids    = torch.zeros(1, 10, dtype=torch.long).to(DEVICE)
dummy_mask   = torch.ones(1, 10, dtype=torch.long).to(DEVICE)
dummy_prefix = torch.zeros(1, PREFIX_LEN, EMBED_DIM, dtype=torch.float32).to(DEVICE)

out_path = f'{ONNX_DIR}/encoder_shared.onnx'

torch.onnx.export(
    shared_encoder,
    (dummy_ids, dummy_mask, dummy_prefix),
    out_path,
    input_names  = ['input_ids', 'attention_mask', 'prefix_embeds'],
    output_names = ['last_hidden_state', 'extended_attention_mask'],
    dynamic_axes = {
        'input_ids'               : {0: 'batch', 1: 'src_len'},
        'attention_mask'          : {0: 'batch', 1: 'src_len'},
        'prefix_embeds'           : {0: 'batch'},
        'last_hidden_state'       : {0: 'batch', 1: 'seq_len'},
        'extended_attention_mask' : {0: 'batch', 1: 'seq_len'},
    },
    opset_version=17,
    do_constant_folding=True,
    dynamo=False,                # legacy TorchScript exporter
    export_params=True,
)

onnx.checker.check_model(out_path)
sz = os.path.getsize(out_path) / 1e6
print(f'\n  encoder_shared.onnx  {sz:.1f} MB  ✅')
print('\n✅  Cell 14 done.')


In [ ]:
# ── CELL 15 — Export 4 decoders (LoRA merged per persona) ────────────────────
# For each persona:
#   1. Temporarily replace LoRA layers with merged nn.Linear
#      (W_new = W_base + scale * A[pid] @ B[pid])
#   2. Export that decoder
#   3. Restore LoRA layers for the next persona
# Result: 4 decoder_{persona}.onnx files. At inference, select the one
# matching the active persona — no LoRA routing needed at runtime.

import torch
import torch.nn as nn
import onnx

class DecoderWrapper(nn.Module):
    def __init__(self, backbone):
        super().__init__()
        self.backbone = backbone
    def forward(self, encoder_hidden_states, encoder_attention_mask, decoder_input_ids):
        out = self.backbone(
            decoder_input_ids=decoder_input_ids,
            encoder_outputs=(encoder_hidden_states,),
            attention_mask=encoder_attention_mask,
        )
        return out.logits


def _merge_lora_for_persona(lora_modules, pid):
    undo = []
    for name, lora_mod in lora_modules:
        A = lora_mod.A[pid].detach()
        B = lora_mod.B[pid].detach()
        delta = (A @ B) * lora_mod.scale
        W_merged = lora_mod.base.weight.detach() + delta.T

        merged = nn.Linear(
            in_features=lora_mod.base.in_features,
            out_features=lora_mod.base.out_features,
            bias=(lora_mod.base.bias is not None),
        ).to(lora_mod.base.weight.device)
        merged.weight.data = W_merged
        if lora_mod.base.bias is not None:
            merged.bias.data = lora_mod.base.bias.detach().clone()

        parts = name.split('.')
        parent = backbone
        for p in parts[:-1]:
            parent = getattr(parent, p)
        child_name = parts[-1]

        undo.append((parent, child_name, lora_mod))
        setattr(parent, child_name, merged)
    return undo

def _restore_lora(undo_list):
    for parent, child_name, original in undo_list:
        setattr(parent, child_name, original)


seq_len     = PREFIX_LEN + 10
dummy_enc_hs   = torch.zeros(1, seq_len, EMBED_DIM).to(DEVICE)
dummy_enc_mask = torch.ones(1, seq_len, dtype=torch.long).to(DEVICE)
dummy_dec_ids  = torch.zeros(1, 5, dtype=torch.long).to(DEVICE)

for pid, pname in PERSONAS.items():
    undo = _merge_lora_for_persona(model.lora_modules, pid)
    try:
        dec_wrapper = DecoderWrapper(backbone).to(DEVICE).eval()
        dec_path = f'{ONNX_DIR}/decoder_{pname}.onnx'

        torch.onnx.export(
            dec_wrapper,
            (dummy_enc_hs, dummy_enc_mask, dummy_dec_ids),
            dec_path,
            input_names  = ['encoder_hidden_states', 'encoder_attention_mask', 'decoder_input_ids'],
            output_names = ['logits'],
            dynamic_axes = {
                'encoder_hidden_states'  : {0: 'batch', 1: 'enc_seq_len'},
                'encoder_attention_mask' : {0: 'batch', 1: 'enc_seq_len'},
                'decoder_input_ids'      : {0: 'batch', 1: 'dec_seq_len'},
                'logits'                 : {0: 'batch', 1: 'dec_seq_len'},
            },
            opset_version=17,
            do_constant_folding=True,
            dynamo=False,
            export_params=True,
        )

        # Inline external weights so each .onnx is self-contained
        m = onnx.load(dec_path, load_external_data=True)
        onnx.save(m, dec_path, save_as_external_data=False)
        if os.path.exists(dec_path + '.data'):
            os.remove(dec_path + '.data')

        onnx.checker.check_model(dec_path)
        sz = os.path.getsize(dec_path) / 1e6
        print(f'  decoder_{pname}.onnx  {sz:.1f} MB  ✅')
    finally:
        _restore_lora(undo)

print('\n✅  Cell 15 done — 4 decoders exported and inlined.')


In [ ]:
# ── CELL 16 — INT8 dynamic quantization → QUANT_DIR_INT8 ─────────────────────
# Writes to a SEPARATE directory so it doesn't clobber the FP16 build.

from onnxruntime.quantization import quantize_dynamic, QuantType
import glob, shutil, onnx, os

# Clean target dir
for path in glob.glob(f'{QUANT_DIR_INT8}/*'):
    if os.path.isfile(path):
        os.remove(path)

onnx_files = sorted(glob.glob(f'{ONNX_DIR}/*.onnx'))
print(f'Files to quantize (INT8): {len(onnx_files)}')

for src_path in onnx_files:
    fname    = os.path.basename(src_path)
    dst_path = f'{QUANT_DIR_INT8}/{fname}'

    quantize_dynamic(
        model_input    = src_path,
        model_output   = dst_path,
        weight_type    = QuantType.QInt8,
        per_channel    = False,
        reduce_range   = False,
    )

    m = onnx.load(dst_path)
    int8_count = sum(1 for n in m.graph.node if n.op_type == 'MatMulInteger')
    orig_mb    = os.path.getsize(src_path) / 1e6
    quant_mb   = os.path.getsize(dst_path) / 1e6
    reduction  = (1 - quant_mb / orig_mb) * 100
    status = '✅' if int8_count > 0 else '❌  NO INT8 OPS'
    print(f'  {fname:30s}  {orig_mb:.1f}MB → {quant_mb:.1f}MB  '
          f'({reduction:.0f}% smaller)  MatMulInt8={int8_count}  {status}')

# Copy persona_prefixes.npy and tokenizer
shutil.copy(f'{ONNX_DIR}/persona_prefixes.npy', f'{QUANT_DIR_INT8}/persona_prefixes.npy')
tokenizer.save_pretrained(QUANT_DIR_INT8)

# Remove HF model files that the tokenizer save sometimes drops
for fname in ['model.safetensors', 'pytorch_model.bin', 'README.md', '.gitattributes']:
    p = f'{QUANT_DIR_INT8}/{fname}'
    if os.path.exists(p):
        os.remove(p)

total_mb = sum(os.path.getsize(f'{QUANT_DIR_INT8}/{f}') for f in os.listdir(QUANT_DIR_INT8)) / 1e6
print(f'\nINT8 dir total size: {total_mb:.0f} MB')
print(f'  → {QUANT_DIR_INT8}')
print('\n✅  Cell 16 done.')


## 6. ONNX Runtime Inference & Validation

In [ ]:
import onnxruntime as ort
import numpy as np
import re

def build_ort_runtime(quant_dir, fp16=False):
    so = ort.SessionOptions()
    so.intra_op_num_threads = 2
    so.inter_op_num_threads = 1
    so.graph_optimization_level = ort.GraphOptimizationLevel.ORT_ENABLE_ALL

    enc_session = ort.InferenceSession(f'{quant_dir}/encoder_shared.onnx', sess_options=so)
    persona_prefixes = np.load(f'{quant_dir}/persona_prefixes.npy')

    dec_sessions = {}
    for pid, pname in PERSONAS.items():
        path = f'{quant_dir}/decoder_{pname}.onnx'
        dec_sessions[pid] = ort.InferenceSession(path, sess_options=so)

    stop_ids = set()
    for tok in ['</s>', '<pad>', '<2hi>']:
        tid = tokenizer.convert_tokens_to_ids(tok)
        if tid is not None and tid != tokenizer.unk_token_id:
            stop_ids.add(tid)
    if tokenizer.eos_token_id is not None:
        stop_ids.add(tokenizer.eos_token_id)

    # Read expected dtypes directly from model metadata
    def _get_np_dtype(ort_type_str):
        if 'float16' in ort_type_str:
            return np.float16
        elif 'int64' in ort_type_str:
            return np.int64
        else:
            return np.float32

    enc_input_dtypes = {inp.name: _get_np_dtype(inp.type) for inp in enc_session.get_inputs()}
    dec_input_dtypes = {inp.name: _get_np_dtype(inp.type) for inp in dec_sessions[0].get_inputs()}

    def _devanagari_only(text):
        return ''.join(
            ch for ch in text
            if '\u0900' <= ch <= '\u097f' or ch in ' \t।?!,'
        ).strip()

    def _first_sentence(text):
        m = re.search(r'[।?!]', text)
        return text[:m.end()].strip() if m else text.strip()

    def _adaptive_max_new_tokens(text, persona_id):
        n = len(tokenizer.encode(text, add_special_tokens=False))
        mult = 2.2 if persona_id in (0, 3) else 2.0
        return max(15, min(35, int(n * mult) + 5))

    def rewrite_ort(text, persona_id, max_new_tokens=None):
        if max_new_tokens is None:
            max_new_tokens = _adaptive_max_new_tokens(text, persona_id)

        enc_in = tokenizer(
            [f'{SRC_LANG} {text}'],
            max_length=MAX_SRC_LEN, truncation=True, return_tensors='np',
        )

        enc_out = enc_session.run(
            None,
            {
                'input_ids'      : enc_in['input_ids'].astype(enc_input_dtypes.get('input_ids', np.int64)),
                'attention_mask' : enc_in['attention_mask'].astype(enc_input_dtypes.get('attention_mask', np.int64)),
                'prefix_embeds'  : persona_prefixes[persona_id:persona_id+1].astype(enc_input_dtypes.get('prefix_embeds', np.float32)),
            }
        )
        hidden_states = enc_out[0].astype(dec_input_dtypes.get('encoder_hidden_states', np.float32))
        ext_mask      = enc_out[1].astype(dec_input_dtypes.get('encoder_attention_mask', np.int64))

        bos_id  = tokenizer.convert_tokens_to_ids(TGT_LANG)
        dec_ids = np.array([[bos_id]], dtype=np.int64)
        generated = []

        for _ in range(max_new_tokens):
            logits = dec_sessions[persona_id].run(
                None,
                {
                    'encoder_hidden_states'  : hidden_states,
                    'encoder_attention_mask' : ext_mask,
                    'decoder_input_ids'      : dec_ids,
                }
            )[0]

            next_id = int(np.argmax(logits[0, -1, :]))
            if next_id in stop_ids:
                break
            generated.append(next_id)
            dec_ids = np.concatenate([dec_ids, [[next_id]]], axis=1)

        raw   = tokenizer.decode(generated, skip_special_tokens=True)
        clean = _devanagari_only(raw)
        return _first_sentence(clean)

    return rewrite_ort

print('✅  Cell 18 done — build_ort_runtime() defined.')

In [ ]:
# ── INT8 Quality Check — chrF++ vs PyTorch baseline ──────────────────────────
import sacrebleu
import time
import os

N_PER_PERSONA = 30

print('Loading INT8 ONNX runtime...')
rewrite_ort = build_ort_runtime(QUANT_DIR_INT8, fp16=False)

print('\n── chrF++ per persona (INT8 vs PyTorch baseline) ──')
for pid, pname in PERSONAS.items():
    persona_rows = [r for r in val_rows if r[2] == pid][:N_PER_PERSONA]
    ort_hyps = [rewrite_ort(n, pid) for (n, s, p) in persona_rows]
    refs     = [s for (n, s, p) in persona_rows]
    chrf = sacrebleu.corpus_chrf(ort_hyps, [refs]).score
    drop = PT_BASELINE_CHRF[pid] - chrf
    status = '✅' if drop < 5.0 else '⚠️'
    print(f'  {pname:14s}  PT={PT_BASELINE_CHRF[pid]:.2f}  INT8={chrf:.2f}  drop={drop:+.2f}  {status}')

# Size
total_mb = sum(os.path.getsize(f'{QUANT_DIR_INT8}/{f}') for f in os.listdir(QUANT_DIR_INT8)) / 1e6

# Latency
latencies = []
for s in SPOT_CHECK:
    t0 = time.perf_counter()
    rewrite_ort(s, 0)
    latencies.append((time.perf_counter() - t0) * 1000)
latencies.sort()
p50 = latencies[len(latencies)//2]
p95 = latencies[int(len(latencies)*0.95)]

print(f'\n  Total size     : {total_mb:.0f} MB')
print(f'  Latency P50/P95: {p50:.0f} / {p95:.0f} ms')
print('\n✅  INT8 quality check done.')

## 7. Package & Download

In [ ]:
import shutil, os
from google.colab import files

zip_name = '/content/indicbart_hindi_int8'
print('Creating zip (INT8)...')
zip_file = shutil.make_archive(zip_name, 'zip', QUANT_DIR_INT8)
zip_mb = os.path.getsize(zip_file) / 1e6
print(f'✅  {zip_file}  ({zip_mb:.0f} MB)')

print('Starting download...')
files.download(zip_file)

print()
print('── Contents ──')
print(f'{os.path.basename(QUANT_DIR_INT8)}/')
for f in sorted(os.listdir(QUANT_DIR_INT8)):
    sz = os.path.getsize(f'{QUANT_DIR_INT8}/{f}') / 1e6
    print(f'  {f:40s}  {sz:.1f} MB')

### Verify the Zip

In [ ]:
import zipfile
import os
import numpy as np
import onnxruntime as ort

# Unzip to a fresh test directory
TEST_DIR = '/content/test_int8_unzipped'
if os.path.exists(TEST_DIR):
    shutil.rmtree(TEST_DIR)
os.makedirs(TEST_DIR)

with zipfile.ZipFile('/content/indicbart_hindi_int8.zip', 'r') as z:
    z.extractall(TEST_DIR)

print('Extracted files:')
for f in sorted(os.listdir(TEST_DIR)):
    sz = os.path.getsize(f'{TEST_DIR}/{f}') / 1e6
    print(f'  {f:40s}  {sz:.1f} MB')

# Load and run inference from the unzipped build
print('\n── Loading from unzipped build ──')
rewrite_test = build_ort_runtime(TEST_DIR, fp16=False)

print('\n── Spot check all 4 personas ──')
test_inputs = ['तुम कहाँ हो?', 'खाना खा लिया?', 'मुझे माफ करो।']

for pid, pname in PERSONAS.items():
    print(f'\n  [{pname}]')
    for s in test_inputs:
        out = rewrite_test(s, pid)
        print(f'    {s}  →  {out}')

print('\n✅ INT8 zip verified — model loads and generates from the unzipped build.')

### Extended Spot Check

In [ ]:
test_sentences = [
    'मैं बाहर जा रहा हूँ।',
    'पानी पिला दो।',
    'आज बहुत थक गया हूँ।',
    'कल मिलते हैं।',
    'तुमने फोन क्यों नहीं उठाया?',
    'मुझे बहुत बुखार है।',
    'पैसे भेज दो।',
    'मैं तुमसे बात नहीं करना चाहता।',
    'आज मौसम बहुत अच्छा है।',
    'तुम मुझसे झूठ बोल रहे हो।',
    'मैं खुश हूँ।',
    'दवाई ले ली?',
    'मुझे अकेला छोड़ दो।',
    'बहुत देर हो गई, सो जाओ।',
    'तुम्हारी याद आ रही है।',
]

for pid, pname in PERSONAS.items():
    print(f'\n{"="*60}')
    print(f'  {pname.upper()}')
    print('='*60)
    for s in test_sentences:
        out = rewrite_test(s, pid)
        print(f'  {s}  →  {out}')

## 8. Runtime Inference ReferenceAfter downloading and unzipping, use this code to load and run the model on any device:```pythonimport onnxruntime as ortimport numpy as npimport refrom transformers import AutoTokenizerMODEL_DIR = './indicbart_hindi_int8'PERSONAS  = {0: 'mother', 1: 'stranger', 2: 'friend', 3: 'gf_wife'}SRC_LANG  = '<2hi>'TGT_LANG  = '<2hi>'tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR, trust_remote_code=True)so = ort.SessionOptions()so.intra_op_num_threads = 2enc_session = ort.InferenceSession(f'{MODEL_DIR}/encoder_shared.onnx', sess_options=so)persona_prefixes = np.load(f'{MODEL_DIR}/persona_prefixes.npy')dec_sessions = {    pid: ort.InferenceSession(f'{MODEL_DIR}/decoder_{pname}.onnx', sess_options=so)    for pid, pname in PERSONAS.items()}stop_ids = {tokenizer.eos_token_id, tokenizer.convert_tokens_to_ids('</s>')}def rewrite(text, persona_id, max_new_tokens=30):    enc_in = tokenizer([f'{SRC_LANG} {text}'], max_length=64, truncation=True, return_tensors='np')    enc_out = enc_session.run(None, {        'input_ids':      enc_in['input_ids'].astype(np.int64),        'attention_mask': enc_in['attention_mask'].astype(np.int64),        'prefix_embeds':  persona_prefixes[persona_id:persona_id+1].astype(np.float32),    })    hidden_states, ext_mask = enc_out[0], enc_out[1]    bos_id  = tokenizer.convert_tokens_to_ids(TGT_LANG)    dec_ids = np.array([[bos_id]], dtype=np.int64)    generated = []    for _ in range(max_new_tokens):        logits = dec_sessions[persona_id].run(None, {            'encoder_hidden_states':  hidden_states,            'encoder_attention_mask': ext_mask.astype(np.int64),            'decoder_input_ids':      dec_ids,        })[0]        next_id = int(np.argmax(logits[0, -1, :]))        if next_id in stop_ids: break        generated.append(next_id)        dec_ids = np.concatenate([dec_ids, [[next_id]]], axis=1)    raw = tokenizer.decode(generated, skip_special_tokens=True)    clean = ''.join(c for c in raw if '\u0900' <= c <= '\u097f' or c in ' \t।?!,').strip()    m = re.search(r'[।?!]', clean)    return clean[:m.end()].strip() if m else clean# Exampleneutral = 'तुम कहाँ हो?'for pid, pname in PERSONAS.items():    print(f'[{pname:10s}] {rewrite(neutral, pid)}')```